# Ordered Logistic Regression Results: FAIRˆ² Dataset Exploration with `mlcroissant`

This notebook demonstrates how to explore the FAIRˆ² dataset on adoption predictors of knowledge in rangeland management using the [`mlcroissant`](https://github.com/mlcommons/croissant) library and the [Croissant schema](https://mlcommons.org/croissant/). The notebook covers dataset loading, overview, extraction, and exploratory analysis. All dataset entities (record sets, fields, columns) are referenced by their Croissant `@id`s for clarity and reproducibility.

### Dataset Source
The dataset is described by a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Install mlcroissant if needed
!pip install -U mlcroissant

## 1. Data Loading
Load both metadata and records using `mlcroissant`.   
We first examine the dataset and its full metadata. 

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load Dataset
dataset = mlc.Dataset(croissant_url)

# Print dataset-level metadata fields
metadata = dataset.metadata  # Not treated as dict
print(f"Dataset name: {getattr(metadata, 'name', '<no name>')}")
print(f"Description: {getattr(metadata, 'description', '<no description>')}")
print(f"Version: {getattr(metadata, 'version', '<no version>')}")
print(f"Published: {getattr(metadata, 'datePublished', '<no date>')}")

## 2. Data Overview
The Croissant dataset contains one or more record sets. We list all available record sets and their fields, referencing entities by their full Croissant `@id`.

We will print all record set `@id`s and under each, all field `@id`s.

In [ ]:
# View all record sets and their fields using Croissant @id
record_sets = dataset.metadata.record_sets  # This is a list of RecordSet objects
if not record_sets:
    print("No record sets found in this dataset.")
else:
    print("Available record sets and their fields (by @id):\n")
    for rs in record_sets:
        print(f"- RecordSet @id: {rs.id}")
        if hasattr(rs, 'fields'):
            for field in rs.fields:
                print(f"    - Field @id: {field.id} | name: {getattr(field, 'name', '')}")
        else:
            print("    (No fields defined)")

## 3. Data Extraction

We load the data from each record set into a Pandas DataFrame, referencing record set and fields by their full `@id`. Usually, each record set is a structured table in the underlying data files.


In [ ]:
# Extract data from all record sets (referenced by @id)

import collections

if not record_sets:
    print("No record sets to extract from.")
    dataframes = {}
else:
    dataframes = {}  # Dict mapping record set @id -> DataFrame
    record_set_ids = [rs.id for rs in record_sets]
    for record_set_id in record_set_ids:
        print(f"Loading record set: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        if records and isinstance(records[0], collections.abc.Mapping):
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded DataFrame with {df.shape[0]} rows and {df.shape[1]} columns.")
        else:
            print(f"No records found for record set {record_set_id}. Skipping.")
    # For demonstration, print columns from the first non-empty DataFrame
    first_nonempty = [rid for rid, df in dataframes.items() if not df.empty]
    if first_nonempty:
        print(f"\nColumns in {first_nonempty[0]}: {dataframes[first_nonempty[0]].columns.tolist()}")
        display_df = dataframes[first_nonempty[0]].head()
        display(display_df)
    else:
        print("No non-empty DataFrames loaded.")

## 4. Exploratory Data Analysis (EDA)

We'll select a numeric field (by its Croissant field `@id`) from one record set and demonstrate basic analysis: filtering, normalization, and grouping by an attribute (by its `@id`). Replace `<fill-in>` sections if you know specific IDs/columns.

In [ ]:
# EDA: pick one record set DataFrame with data
numeric_field_id = None
group_field_id = None
selected_record_set_id = None
for rs in record_sets:
    rsid = rs.id
    df = dataframes.get(rsid, pd.DataFrame())
    if df.empty:
        continue
    # Try to pick first numeric field (int/float type) from Croissant fields
    for field in getattr(rs, 'fields', []):
        if hasattr(field, 'data_type'):
            dt = getattr(field, 'data_type', '')
            if dt in ('schema:Number', 'schema:Float', 'schema:Integer'):
                # Ensure column is present in DataFrame
                if field.id in df.columns:
                    numeric_field_id = field.id
                    selected_record_set_id = rsid
                    break
    # Try to find group field (prefer string/categorical, must not be the numeric field)
    for field in getattr(rs, 'fields', []):
        if getattr(field, 'data_type', '') == 'schema:Text' and field.id != numeric_field_id:
            if field.id in df.columns:
                group_field_id = field.id
                break
    if numeric_field_id and selected_record_set_id:
        break

if not (numeric_field_id and selected_record_set_id):
    print("No suitable numeric field and record set found for EDA.")
else:
    df = dataframes[selected_record_set_id]
    print(f'Using record set: {selected_record_set_id}')
    print(f'Numeric field (@id): {numeric_field_id}')
    if group_field_id:
        print(f'Group field (@id): {group_field_id}')

    # Drop NA for numeric analysis
    df_numeric = df.dropna(subset=[numeric_field_id]).copy()

    # Demonstrate threshold filtering
    if not df_numeric[numeric_field_id].empty:
        # Cast to numeric if possible (if imported as str/object)
        df_numeric[numeric_field_id] = pd.to_numeric(df_numeric[numeric_field_id], errors='coerce')
        # Example: use the 75th percentile as a threshold
        threshold = df_numeric[numeric_field_id].quantile(0.75)
        filtered_df = df_numeric[df_numeric[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (75th percentile):")
        print(filtered_df.head())

        # Normalize the field (z-score)
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by group_field if available
        if group_field_id and group_field_id in filtered_df:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nAverage of {numeric_field_id} grouped by {group_field_id}:")
            print(grouped_df.head())
    else:
        print(f"No non-null values found for {numeric_field_id}.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field, and if available, compare it across groupings.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

if selected_record_set_id and numeric_field_id:
    df = dataframes[selected_record_set_id]
    vals = pd.to_numeric(df[numeric_field_id], errors='coerce').dropna()
    plt.figure(figsize=(8, 4))
    sns.histplot(vals, bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 6))
        # Only take groups with at least 5 records
        group_sizes = df[group_field_id].value_counts()
        major_groups = group_sizes[group_sizes >= 5].index
        for grp in major_groups[:5]:  # plot top 5 groups
            grp_data = pd.to_numeric(df[df[group_field_id] == grp][numeric_field_id], errors='coerce').dropna()
            sns.kdeplot(grp_data, label=str(grp))
        plt.xlabel(numeric_field_id)
        plt.ylabel('Density')
        plt.title(f"{numeric_field_id} distributions by {group_field_id} (top 5 groups)")
        plt.legend(title=group_field_id)
        plt.show()
else:
    print("No numeric field or record set selected for visualization.")

## 6. Conclusion

We have demonstrated how to load and explore the FAIRˆ² dataset using `mlcroissant`, referencing all entities using Croissant `@id`s. This included dataset metadata exploration, dynamic extraction of record sets, and basic EDA including filtering, normalization, and visualization. For further and reproducible analysis, always specify fields and tables by their Croissant `@id`.